# Level 1 · Task 1 — Data Preprocessing for Machine Learning

**Goal:** take the raw `sentiment.csv` dataset and make it fully ready for machine learning.

**What we'll do (as per the task requirements):**
1. Explore the raw data and find quality issues
2. Handle missing data (median fill / dropping)
3. Encode categorical variables (one-hot + label encoding)
4. Normalize/standardize numerical features
5. Split into training and testing sets

*Tools: Python, pandas, scikit-learn*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.width', 120)
pd.set_option('display.max_columns', 30)

df = pd.read_csv('../data/sentiment.csv')
print('shape:', df.shape)
df.head()

In [ ]:
# quick overview of columns and types
df.info()

In [ ]:
# missing values and duplicates
print('Missing values per column:')
print(df.isna().sum())
print('\nDuplicate rows:', df.duplicated().sum())

In [ ]:
# the target column looks messy — let's check
print('unique sentiment labels:', df['Sentiment'].nunique())
print('\na few raw examples:')
print(df['Sentiment'].sample(12, random_state=1).tolist())

In [ ]:
# top 15 most frequent raw labels
top = df['Sentiment'].str.strip().value_counts().head(15)

plt.figure(figsize=(9, 4))
top.plot(kind='bar', color='steelblue')
plt.title('Top 15 raw sentiment labels (before cleaning)')
plt.ylabel('count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Step 1 — Cleaning the data

Issues found in the exploration:
- extra **whitespace** inside string values (`' Twitter  '`, `' Positive  '`)
- two useless **index columns** (`Unnamed: 0` and the first unnamed one)
- **missing values** in some numeric columns
- **~180 different sentiment labels**, some being stray phrases — needs consolidation

In [ ]:
# strip whitespace from all string columns
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

# drop redundant index columns
df = df.drop(columns=['Unnamed: 0'])
df = df.loc[:, ~df.columns.str.contains('^Unnamed')]

# drop exact duplicate rows
before = len(df)
df = df.drop_duplicates()
print(f'dropped {before - len(df)} duplicate rows -> {len(df)} rows left')

In [ ]:
# handle missing values
# - numeric columns -> fill with median (robust to outliers)
# - rows missing the target label -> drop
num_cols = ['Retweets', 'Likes']
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

df = df.dropna(subset=['Sentiment'])
df['Text'] = df['Text'].fillna('')
df['Hashtags'] = df['Hashtags'].fillna('')

print('missing values after cleaning:')
print(df.isna().sum().sum(), 'total')

In [ ]:
# consolidate the messy ~180 labels into 3 clean classes
POSITIVE = ['positive', 'joy', 'happy', 'excit', 'content', 'gratit', 'love', 'proud',
            'pride', 'awe', 'euphor', 'enthus', 'seren', 'calm', 'hope', 'inspir',
            'determin', 'amuse', 'tender', 'affect', 'ador', 'kind', 'compassion',
            'delight', 'bliss', 'festive', 'celebr', 'charm', 'ecstas', 'triumph',
            'success', 'wonder', 'amaz', 'thrill', 'zest', 'relief', 'rejuv',
            'resilien', 'tranquil', 'solace', 'playful', 'energ', 'spark', 'dazzle',
            'enchant', 'fulfill', 'touched', 'heartwarm', 'empower', 'satisf',
            'optimis', 'motivat', 'bless', 'freedom', 'friend', 'roman', 'creativ',
            'confiden', 'magic', 'breakthrough', 'elat', 'grateful']

NEGATIVE = ['negative', 'despair', 'loneli', 'grief', 'sad', 'sorrow', 'anger', 'hate',
            'frustrat', 'fear', 'anxie', 'melanchol', 'embarrass', 'regret', 'shame',
            'betray', 'bitter', 'disgust', 'devast', 'jealous', 'resent', 'envy',
            'heartbreak', 'desp', 'dark', 'ruin', 'loss', 'lost', 'exhaust', 'suffer',
            'pressur', 'intimid', 'helpless', 'miser', 'pain', 'rage', 'storm',
            'overwhelm', 'isolat', 'broken', 'bad']

def simplify_sentiment(label):
    l = str(label).lower()
    if any(k in l for k in NEGATIVE):
        return 'Negative'
    if any(k in l for k in POSITIVE):
        return 'Positive'
    return 'Neutral'

df['Sentiment'] = df['Sentiment'].apply(simplify_sentiment)
df['Sentiment'].value_counts()

In [ ]:
# labelled distribution after cleaning
df['Sentiment'].value_counts().plot(kind='bar', color=['tomato', 'seagreen', 'gray'])
plt.title('Sentiment distribution (after cleaning)')
plt.ylabel('count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Step 2 — Feature engineering & encoding categorical variables

- derive simple numeric features from the raw text fields
- **one-hot encode** `Platform` and `Country`
- **label encode** the target `Sentiment`

In [ ]:
# simple numeric features from text
df['text_length'] = df['Text'].str.len()
df['hashtag_count'] = df['Hashtags'].str.count('#').fillna(0).astype(int)

# drop columns we no longer need (raw text / identifiers)
df = df.drop(columns=['Text', 'User', 'Timestamp', 'Hashtags'])

# drop constant columns (a column with a single value carries no information)
constant_cols = [c for c in df.columns if df[c].nunique() == 1]
if constant_cols:
    print('dropping constant columns:', constant_cols)
    df = df.drop(columns=constant_cols)

df.head()

In [ ]:
# one-hot encode categorical features
cat_cols = ['Platform', 'Country']
df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)

# label encode the target
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['Sentiment'] = le.fit_transform(df['Sentiment'])

print('class mapping:', dict(zip(le.classes_, le.transform(le.classes_))))
print('\nshape after encoding:', df.shape)
df.head()

## Step 3 — Normalize / standardize numerical features

Scaling the continuous features to zero mean and unit variance so that
features on big scales (likes, retweets) don't dominate the model.

In [ ]:
from sklearn.preprocessing import StandardScaler

scale_cols = ['Retweets', 'Likes', 'text_length', 'hashtag_count']

print('BEFORE scaling:')
print(df[scale_cols].describe().loc[['mean', 'std']].round(2))

scaler = StandardScaler()
df[scale_cols] = scaler.fit_transform(df[scale_cols])

print('\nAFTER scaling:')
print(df[scale_cols].describe().loc[['mean', 'std']].round(2))